# AuditLens - fine-tuning the review-comment model

We do not call a hosted language model. This notebook trains our own.

**Base model:** `google/flan-t5-base` (250M parameters) - small enough to run on CPU
inside our container, and strong at structured text generation.

**What it learns:** to turn a structured description of what our deterministic engine
computed into a professional review comment. The rules are the teacher; the model only
learns how to *express* something already established as true. It never invents a figure.

**Runtime:** set *Runtime -> Change runtime type -> T4 GPU* before running.
Training takes roughly 15-25 minutes.


## 1. Setup


In [ ]:
!pip install -q transformers datasets accelerate sentencepiece evaluate rouge-score


In [ ]:
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
assert torch.cuda.is_available(), 'Enable the GPU: Runtime > Change runtime type > T4 GPU'


## 2. Training data

Generated by `training/generate_dataset.py` in the repository. Upload `train.jsonl`
and `validation.jsonl` using the file panel on the left.


In [ ]:
import os
from google.colab import files

if not os.path.exists('train.jsonl'):
    print('Upload train.jsonl and validation.jsonl')
    files.upload()


In [ ]:
from datasets import load_dataset

raw = load_dataset('json', data_files={
    'train': 'train.jsonl',
    'validation': 'validation.jsonl',
})
print(raw)
print()
print('INPUT :', raw['train'][0]['input'][:200])
print('TARGET:', raw['train'][0]['target'][:200])


## 3. Model and tokeniser


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = 'google/flan-t5-base'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

params = sum(p.numel() for p in model.parameters())
print(f'{MODEL_NAME}: {params/1e6:.0f}M parameters')


### Baseline - what the model writes *before* training

Run this now and keep the output. The before/after comparison is the clearest
evidence that training did something, and it belongs in the presentation.


In [ ]:
def generate(text, mdl=None, max_new_tokens=110):
    mdl = mdl or model
    device = next(mdl.parameters()).device
    enc = tokenizer('Write a financial review comment. ' + text,
                    return_tensors='pt', truncation=True, max_length=512).to(device)
    out = mdl.generate(**enc, max_new_tokens=max_new_tokens, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

sample = raw['validation'][0]
print('INPUT   :', sample['input'][:200])
print()
print('BEFORE  :', generate(sample['input']))
print()
print('EXPECTED:', sample['target'])


## 4. Tokenise


In [ ]:
PREFIX = 'Write a financial review comment. '
MAX_INPUT, MAX_TARGET = 512, 160

def preprocess(batch):
    enc = tokenizer([PREFIX + t for t in batch['input']],
                    max_length=MAX_INPUT, truncation=True)
    labels = tokenizer(text_target=batch['target'],
                       max_length=MAX_TARGET, truncation=True)
    enc['labels'] = labels['input_ids']
    return enc

tokenised = raw.map(preprocess, batched=True,
                    remove_columns=raw['train'].column_names)
print(tokenised)


## 5. Train


In [ ]:
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

args = Seq2SeqTrainingArguments(
    output_dir='auditlens-reviewer',
    learning_rate=3e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    predict_with_generate=True,
    logging_steps=25,
    fp16=False,          # T5 is unstable in fp16; use bf16 or fp32
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenised['train'],
    eval_dataset=tokenised['validation'],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

history = trainer.train()


### The loss curve

Put this chart in the deck. It is the simplest possible evidence that a model was
actually trained rather than downloaded.


In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_loss = [(l['epoch'], l['loss']) for l in logs if 'loss' in l]
eval_loss  = [(l['epoch'], l['eval_loss']) for l in logs if 'eval_loss' in l]

plt.figure(figsize=(7, 4))
plt.plot(*zip(*train_loss), label='training loss')
plt.plot(*zip(*eval_loss), label='validation loss', marker='o')
plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('AuditLens review-comment model')
plt.legend(); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()


## 6. Evaluation

Two measures, and the second is the one that matters.

**ROUGE** - does the wording resemble a reviewer's?

**Groundedness** - does every number in the generated comment appear in the input?
This is our central claim made measurable. A model that invents a figure fails here,
however fluent it sounds.


In [ ]:
import evaluate

rouge = evaluate.load('rouge')

val = raw['validation']
preds = [generate(x) for x in val['input']]

scores = rouge.compute(predictions=preds, references=val['target'])
for k, v in scores.items():
    print(f'{k:<12} {v:.4f}')


In [ ]:
import re

NUM = re.compile(r'-?\d[\d,]*\.?\d*')

def numbers(text):
    return {n.replace(',', '') for n in NUM.findall(text)}

grounded, hallucinated, total_nums = 0, 0, 0
offenders = []

for inp, pred in zip(val['input'], preds):
    src = numbers(inp)
    out = numbers(pred)
    invented = {n for n in out if n not in src}
    total_nums += len(out)
    hallucinated += len(invented)
    if invented:
        offenders.append((pred, invented))
    else:
        grounded += 1

print(f'comments fully grounded : {grounded}/{len(preds)} ({grounded/len(preds):.1%})')
print(f'numbers generated       : {total_nums}')
print(f'numbers not in the input: {hallucinated} ({hallucinated/max(total_nums,1):.2%})')
print()
for pred, invented in offenders[:3]:
    print('INVENTED', invented, '->', pred[:160])


> **Read this honestly.** Some 'invented' numbers are legitimate - a year written as
> *FY2023* when the input said *year: 2023*, or a percentage expressed differently.
> Look at the offenders before quoting the figure, and say in the presentation which
> are real hallucinations and which are formatting.
>
> In production this is not the only defence: the Evidence Agent supplies only computed
> figures, and the runtime groundedness harness re-checks every number.


### After training


In [ ]:
for i in range(3):
    s = raw['validation'][i]
    print('INPUT   :', s['input'][:180])
    print('AFTER   :', generate(s['input']))
    print('EXPECTED:', s['target'][:200])
    print('-' * 100)


## 7. Save and download

The saved folder goes into the container image, so the application loads the model
from disk and needs no internet connection at runtime.


In [ ]:
SAVE_DIR = 'auditlens-reviewer-final'

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

!du -sh {SAVE_DIR}
!ls -la {SAVE_DIR}


In [ ]:
!zip -qr auditlens-reviewer-final.zip {SAVE_DIR} loss_curve.png

from google.colab import files
files.download('auditlens-reviewer-final.zip')


---

## What to record for the presentation

| Figure | Where it comes from |
|---|---|
| Base model and parameter count | Section 3 |
| Training and validation examples | Section 2 |
| Final training / validation loss | Section 5 |
| Loss curve image | `loss_curve.png` |
| ROUGE-1 / ROUGE-L | Section 6 |
| Groundedness rate | Section 6 |
| Before / after comparison | Sections 3 and 6 |

Unzip `auditlens-reviewer-final.zip` into `models/reviewer/` in the repository.
**Do not commit the weights to git** - they are around 1 GB. Put them in S3 and have
the Docker build pull them in, or attach them to a GitHub release.
